In [1]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("text2text-generation", model="lvcalucioli/flan-t5-large__question-answering", device = "cuda")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/821 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

Device set to use cuda


In [2]:
from itertools import permutations
# Define the question and options
question = "What is the value of p in 24 = 2p?"
options = [ "\nA) p = 4", "\nB) p = 8", "\nC) p = 12", "\nD) p = 24" ]
# options = ["\nA) 1", "\nB) 4", "\nC) 2", "\nD) 6"]
# options = ["1", "4", "2", "6"]

# Generate all permutations of options
option_permutations = permutations(options)



# Generate responses for all permutations
for perm in option_permutations:
    permuted_options = ", ".join(perm)
    input_text = f"{question} {permuted_options}"
    response = pipe(input_text)
    print(f"Input: {input_text}")
    print(f"Output: {response[0]['generated_text']}")
    print("-" * 80)

Input: What is the value of p in 24 = 2p? 
A) p = 4, 
B) p = 8, 
C) p = 12, 
D) p = 24
Output: D
--------------------------------------------------------------------------------
Input: What is the value of p in 24 = 2p? 
A) p = 4, 
B) p = 8, 
D) p = 24, 
C) p = 12
Output: D
--------------------------------------------------------------------------------
Input: What is the value of p in 24 = 2p? 
A) p = 4, 
C) p = 12, 
B) p = 8, 
D) p = 24
Output: D
--------------------------------------------------------------------------------
Input: What is the value of p in 24 = 2p? 
A) p = 4, 
C) p = 12, 
D) p = 24, 
B) p = 8
Output: D
--------------------------------------------------------------------------------
Input: What is the value of p in 24 = 2p? 
A) p = 4, 
D) p = 24, 
B) p = 8, 
C) p = 12
Output: D
--------------------------------------------------------------------------------
Input: What is the value of p in 24 = 2p? 
A) p = 4, 
D) p = 24, 
C) p = 12, 
B) p = 8
Output: D
-------------

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Input: What is the value of p in 24 = 2p? 
B) p = 8, 
C) p = 12, 
A) p = 4, 
D) p = 24
Output: D
--------------------------------------------------------------------------------
Input: What is the value of p in 24 = 2p? 
B) p = 8, 
C) p = 12, 
D) p = 24, 
A) p = 4
Output: D
--------------------------------------------------------------------------------
Input: What is the value of p in 24 = 2p? 
B) p = 8, 
D) p = 24, 
A) p = 4, 
C) p = 12
Output: C
--------------------------------------------------------------------------------
Input: What is the value of p in 24 = 2p? 
B) p = 8, 
D) p = 24, 
C) p = 12, 
A) p = 4
Output: D
--------------------------------------------------------------------------------
Input: What is the value of p in 24 = 2p? 
C) p = 12, 
A) p = 4, 
B) p = 8, 
D) p = 24
Output: D
--------------------------------------------------------------------------------
Input: What is the value of p in 24 = 2p? 
C) p = 12, 
A) p = 4, 
D) p = 24, 
B) p = 8
Output: D
-------------

In [3]:
# Load model directly
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("lvcalucioli/flan-t5-large__question-answering")
model = AutoModelForSeq2SeqLM.from_pretrained("lvcalucioli/flan-t5-large__question-answering")

In [4]:
# from itertools import permutations
# # Define the question and options
# question = "Find the degree for the given field extension Q(sqrt(2), sqrt(3), sqrt(18)) over Q."
# options = ["\nA) 1", "\nB) 4", "\nC) 2", "\nD) 6"]
# # Generate all permutations of options
# option_permutations = permutations(options)



# # Generate responses for all permutations
# for perm in option_permutations:
#     permuted_options = ", ".join(perm)
#     input_text = f"{question} {permuted_options}"
#     # print(len(input_text))
#     if len(tokenizer(input_text).input_ids) ==45:
#       print("*",input_text)
#     else:
#       print(input_text)




In [5]:
option_permutations = permutations(options)
cross_attentions = []
answer_texts = []
# Process each permutation
for perm in option_permutations:
    # Create the question text with permuted options
    permuted_question = f"{question} {', '.join(perm)}"

    # Tokenize the input
    input_ids = tokenizer(permuted_question, return_tensors="pt").input_ids

    # Initialize decoder input with <pad>
    decoder_input_ids = tokenizer(["<pad>"], return_tensors="pt").input_ids
    # decoder_input_ids = model.prepare_decoder_input_ids_from_labels(input_ids)

    # Initialize an empty list to store predicted tokens
    predicted_tokens = []


    with torch.inference_mode():
        outputs = model(
            input_ids=input_ids,
            decoder_input_ids=decoder_input_ids,
            output_attentions=True,
            output_hidden_states=True
        )


    # Get the next token logits and predicted token
    logits = outputs.logits  # Shape: (batch_size, seq_len, vocab_size)
    print("logits shape", logits.shape)
    next_token_id = logits.argmax(dim=-1)[0]
    # [:, -1, :].argmax(dim=1)  # Select the most likely token



    # Decode the full predicted sequence
    response = tokenizer.decode(next_token_id, skip_special_tokens=True)
    answer_texts.append(response)
    # Print the results
    print(f"Input: {permuted_question}")
    print(f"Output: {response}")
    cross_per_head =  []
    print(outputs.cross_attentions[-1][0][0][0])
    for attn_head in outputs.cross_attentions[-1][0]:

      # print(attn_head[0])
      cross_per_head.append(attn_head[0])
    cross_attentions.append(cross_per_head)
    print("-" * 80)

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


logits shape torch.Size([1, 2, 32128])
Input: What is the value of p in 24 = 2p? 
A) p = 4, 
B) p = 8, 
C) p = 12, 
D) p = 24
Output: D
tensor([0.0031, 0.0229, 0.0210, 0.0336, 0.0048, 0.0210, 0.0056, 0.0029, 0.0077,
        0.0039, 0.0179, 0.0136, 0.0113, 0.0462, 0.0170, 0.0211, 0.0045, 0.0098,
        0.0072, 0.0765, 0.0053, 0.0208, 0.0029, 0.0090, 0.0104, 0.2554, 0.0132,
        0.0209, 0.0058, 0.0152, 0.0137, 0.1496, 0.0285, 0.0216, 0.0149, 0.0331,
        0.0181, 0.0101])
--------------------------------------------------------------------------------
logits shape torch.Size([1, 2, 32128])
Input: What is the value of p in 24 = 2p? 
A) p = 4, 
B) p = 8, 
D) p = 24, 
C) p = 12
Output: D
tensor([0.0028, 0.0225, 0.0205, 0.0292, 0.0046, 0.0206, 0.0051, 0.0031, 0.0070,
        0.0033, 0.0169, 0.0148, 0.0114, 0.0470, 0.0182, 0.0206, 0.0042, 0.0093,
        0.0066, 0.0982, 0.0067, 0.0203, 0.0032, 0.0097, 0.0143, 0.2109, 0.0236,
        0.0203, 0.0068, 0.0210, 0.0123, 0.1814, 0.0150, 0.0212

In [6]:
len(cross_attentions[0])

16

In [7]:
from scipy.spatial.distance import jensenshannon
from scipy.special import rel_entr
import numpy as np
average_divergences_all = []
average_divergences_kl_all = []
cross_attentions =  np.array(cross_attentions)
for cross_attention in cross_attentions:
  permute_divergence = []
  permute_divergence_kl = []
  for head in range(cross_attentions.shape[1]):

    permute_divergence_head = []
    permute_divergence_kl_head = []
    matrix = np.array(cross_attention[head])
    n = cross_attentions.shape[0]


    # Compute pairwise JS divergences
    for i in range(n):
        divergences = []
        kl_divergences = []


        for j in range(n):
            if i != j:  # Exclude self-comparison

                matrix2 =  np.array(cross_attentions[j])
                divergence = jensenshannon(matrix[i], matrix2[head])
                divergences.append(divergence)
                kl_div = np.sum(rel_entr(matrix[i], matrix2[head]))  # KL divergence P || Q
                kl_divergences.append(kl_div)
        permute_divergence_head.append(np.mean(divergences))  # Average divergence for row `i`
        permute_divergence_kl_head.append(np.mean(kl_divergences))
    permute_divergence.append(permute_divergence_head)
    permute_divergence_kl_head.append(permute_divergence_kl_head)
  average_divergences_all.append(permute_divergence)
  average_divergences_kl_all.append(permute_divergence_kl)


average_divergences = np.array(average_divergences_all).mean(axis=1)

average_divergences_kl = np.array(average_divergences_kl_all).mean(axis=1)


# # Find the most different instance (highest average divergence)
# most_different_index = np.argmax(average_divergences)
# all_indices = np.argsort(average_divergences)[::-1]
# print(f"Average divergences: {average_divergences}")
# print(f"Most different instance is at index {most_different_index} with divergence {average_divergences[most_different_index]}")

# for least in all_indices:
#     print(answer_texts[least])

<ipython-input-7-c59282f20f86>:42: RuntimeWarning: Mean of empty slice.
  average_divergences_kl = np.array(average_divergences_kl_all).mean(axis=1)
/usr/local/lib/python3.10/dist-packages/numpy/core/_methods.py:121: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(


In [13]:
cross_attentions.sum(axis = 1).shape

(24, 38)

In [9]:
np.argsort(np.array(average_divergences_kl_all).mean(axis=1))


<ipython-input-9-f3d2d6b06e25>:1: RuntimeWarning: Mean of empty slice.
  np.argsort(np.array(average_divergences_kl_all).mean(axis=1))


array([12, 23, 22, 21, 20, 19, 18, 17, 16, 15, 14, 13,  0, 11, 10,  9,  8,
        7,  6,  5,  4,  3,  2,  1])

In [10]:
print(np.unique(np.array(answer_texts), return_counts=True))

(array(['A', 'C', 'D'], dtype='<U1'), array([ 1,  6, 17]))


In [11]:
(outputs.cross_attentions[0].shape)

torch.Size([1, 16, 2, 38])

In [19]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity


# Compute pairwise cosine similarity
similarity_matrix = cosine_similarity(cross_attentions.sum(axis = 1))

# Similarity matrix is symmetric with diagonal = 1
# print(similarity_matrix)

k  = 24

# Calculate average similarity for each vector
average_similarity = np.mean(similarity_matrix, axis=1)

print(average_similarity)
least_k_indices = np.argsort(average_similarity)[:k]

print(f"Average similarities: {average_similarity}")
print(f"Indices of least {k} different vectors: {least_k_indices}")
print(f"Least {k} different vectors have averages: {average_similarity[least_k_indices]}")
for least in least_k_indices:
    print(answer_texts[least])


[0.97459716 0.9726384  0.9775725  0.97242707 0.97369546 0.97527593
 0.972892   0.9717068  0.9738377  0.9713884  0.9763362  0.9735017
 0.96848726 0.96828175 0.9694585  0.972812   0.9707269  0.965498
 0.9768999  0.9774917  0.97676545 0.97553015 0.97704107 0.9736491 ]
Average similarities: [0.97459716 0.9726384  0.9775725  0.97242707 0.97369546 0.97527593
 0.972892   0.9717068  0.9738377  0.9713884  0.9763362  0.9735017
 0.96848726 0.96828175 0.9694585  0.972812   0.9707269  0.965498
 0.9768999  0.9774917  0.97676545 0.97553015 0.97704107 0.9736491 ]
Indices of least 24 different vectors: [17 13 12 14 16  9  7  3  1 15  6 11 23  4  8  0  5 21 10 20 18 22 19  2]
Least 24 different vectors have averages: [0.965498   0.96828175 0.96848726 0.9694585  0.9707269  0.9713884
 0.9717068  0.97242707 0.9726384  0.972812   0.972892   0.9735017
 0.9736491  0.97369546 0.9738377  0.97459716 0.97527593 0.97553015
 0.9763362  0.97676545 0.9768999  0.97704107 0.9774917  0.9775725 ]
D
D
D
D
D
D
D
D
D
D
D
D


In [ ]:
similarity_matrix[0]

In [15]:
print(answer_texts)


['D', 'D', 'D', 'D', 'D', 'D', 'D', 'D', 'D', 'D', 'C', 'D', 'D', 'D', 'D', 'D', 'D', 'D', 'C', 'C', 'C', 'C', 'C', 'A']


In [ ]:
print(tokenizer.batch_decode(decoder_input_ids))

In [ ]:
import matplotlib.pyplot as plt
cross_attention_2 =[]
for item in cross_attentions:
  cross_attention_2.append(item[:])
  plt.plot(item[:])
  print(item.shape)

In [18]:
average_similarity, answer_texts

(array([0.97459716, 0.9726384 , 0.9775725 , 0.97242707, 0.97369546,
        0.97527593, 0.972892  , 0.9717068 , 0.9738377 , 0.9713884 ,
        0.9763362 , 0.9735017 , 0.96848726, 0.96828175, 0.9694585 ,
        0.972812  , 0.9707269 , 0.965498  , 0.9768999 , 0.9774917 ,
        0.97676545, 0.97553015, 0.97704107, 0.9736491 ], dtype=float32),
 ['D',
  'D',
  'D',
  'D',
  'D',
  'D',
  'D',
  'D',
  'D',
  'D',
  'C',
  'D',
  'D',
  'D',
  'D',
  'D',
  'D',
  'D',
  'C',
  'C',
  'C',
  'C',
  'C',
  'A'])

In [44]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity


# Compute pairwise cosine similarity
average_attention = cross_attentions.sum(axis = 1).mean(axis=0)


print(average_attention.shape, average_attention )

# # Similarity matrix is symmetric with diagonal = 1


average_similarity = np.abs((cross_attentions.sum(axis = 1) - average_attention).sum(axis =0))

print( )

# k  = 24

# # Calculate average similarity for each vector
# average_similarity = np.mean(similarity_matrix, axis=1)

# print(average_similarity)
# least_k_indices = np.argsort(average_similarity)[:k]

print(f"Average similarities: {average_similarity}")
print(f"Indices of least {k} different vectors: {least_k_indices}")
print(f"Least {k} different vectors have averages: {average_similarity[least_k_indices]}")
for least in least_k_indices:
    print(answer_texts[least])


(38,) [0.16829835 1.3059651  1.1967031  0.41251373 0.13746713 1.1999408
 0.14165683 0.1624606  0.29398865 0.06598058 0.23328404 0.28183857
 0.16702087 0.45425844 0.12928121 1.1934663  0.08595541 0.06578267
 0.16347462 0.6557332  0.0937463  1.1844372  0.07174221 0.08077468
 0.19563246 0.81369156 0.11230149 1.1876324  0.06990505 0.08748386
 0.19974315 0.9065445  0.13699092 1.2305359  0.12772207 0.12785265
 0.2870291  0.5711642 ]

Average similarities: [3.1292439e-07 3.4570694e-06 2.3841858e-07 1.0430813e-06 7.0035458e-07
 1.1920929e-07 4.2468309e-07 2.8312206e-07 8.9406967e-08 2.1979213e-07
 2.9802322e-08 2.3841858e-07 1.9371510e-07 4.1723251e-07 1.6391277e-07
 1.3113022e-06 1.1175871e-07 6.7055225e-08 1.4901161e-07 1.4901161e-06
 1.7881393e-07 1.0728836e-06 9.6857548e-08 7.0780516e-08 2.9057264e-07
 5.9604645e-08 2.0116568e-07 1.1920929e-07 3.4645200e-07 2.6077032e-07
 5.3644180e-07 1.0728836e-06 5.5879354e-07 1.1920929e-06 3.3527613e-07
 2.6822090e-07 1.4901161e-07 9.8347664e-07]
Indic